# Qiskit vs Bloqade — same circuits, three runners

For each test circuit we:
1. **Build in Qiskit** → save as OpenQASM 2.0.
2. **Run on Qiskit's AerSimulator** directly (baseline).
3. **Run via `optimize_qasm`** (our Qiskit wrapper) → AerSimulator or fake IBM.
4. **Run via `optimize_qasm_bloqade`** (the Qiskit-bridge version) → Bloqade's `StackMemorySimulator` or `DynamicMemorySimulator`.

Each test uses a **different backend combination** to exercise more of the
wrappers. Here's the matrix:

| Test | qiskit-direct | qiskit-wrap | bloqade |
|---|---|---|---|
| 1. Bell pair | Aer / `statevector` | ideal / `statevector` | `stack` |
| 2. 3-qubit GHZ (Clifford) | Aer / `stabilizer` | ideal / `density_matrix` | `dynamic` |
| 3. Hadamard×4 | Aer / `matrix_product_state` | ideal / `matrix_product_state` | `stack` |
| 4. Mixed rotations | Aer / `statevector` | ideal / `automatic` | `dynamic` |
| 5. Toffoli | Aer / `automatic` | **fake / `fake_manila`** | `stack` |

Test 5 is the only one where the wrapper hits a **noisy fake IBM device** —
expect that column to spread mass off the ideal `111` outcome due to
emulated gate errors.

**Why the bridge version for Bloqade?** The new `prepare_qasm_for_bloqade`
returns a kernel that runs but does not expose measurement outcomes (it
wraps `qasm2.loadfile`, which returns None-valued kernels). The preserved
bridge version generates a `return c` in the source, so `sim.run(kernel)`
gives us bitstrings we can count. For pure-execution smoke-testing without
outcomes, `prepare_qasm_for_bloqade` is the simpler choice; see the final
section.

**Important endian note.** Qiskit labels bitstrings most-significant-qubit
first (so `'10'` means q1=1, q0=0). Bloqade iterates the classical register
in declaration order (so `'10'` means c0=1, c1=0). We normalize both to
`q0 q1 q2 …` ordering for display so the histograms compare apples-to-apples.

In [ ]:
import sys, os, tempfile
from collections import Counter

sys.path.append(os.path.abspath('src'))

import numpy as np
from qiskit import QuantumCircuit, qasm2 as qk_qasm2, transpile
from qiskit_aer import AerSimulator

from qasm_transpile_qiskit import optimize_qasm
from qasm_transpile_bloqade import prepare_qasm_for_bloqade

SHOTS = 2048

In [ ]:
def dump_qasm(qc: QuantumCircuit) -> str:
    '''Write a QuantumCircuit to a temp .qasm file and return the path.'''
    path = tempfile.mktemp(suffix='.qasm')
    qk_qasm2.dump(qc, path)
    return path

def _get_counts_from_databin(databin) -> dict:
    '''SamplerV2 names the counts field after the classical register —
    `meas` for measure_all(), `c` for an explicit creg, etc. Scan the DataBin
    for whichever field exposes get_counts.'''
    for name in dir(databin):
        if name.startswith('_'):
            continue
        val = getattr(databin, name, None)
        if hasattr(val, 'get_counts'):
            return val.get_counts()
    raise RuntimeError(f'No get_counts found on DataBin; fields: {list(databin)}')

def run_qiskit_direct(qc: QuantumCircuit, *, method: str = 'statevector',
                      shots: int = SHOTS) -> Counter:
    '''Qiskit baseline: AerSimulator with the chosen method.

    Available methods include 'statevector' (default, exact), 'density_matrix'
    (mixed-state capable), 'matrix_product_state' (large but low-entanglement),
    'stabilizer' (Clifford-only, fastest), and 'automatic' (Aer picks).
    '''
    sim = AerSimulator(method=method)
    tqc = transpile(qc, sim, optimization_level=0)
    counts = sim.run(tqc, shots=shots).result().get_counts()
    # Qiskit bitstrings are MSB-first — reverse so q0 is first.
    return Counter({k[::-1]: v for k, v in counts.items()})

def run_qiskit_wrapper(qasm_path: str, *, backend_type: str = 'ideal',
                       backend_method: str = 'statevector',
                       shots: int = SHOTS) -> Counter:
    '''Our Qiskit wrapper: optimize_qasm -> SamplerV2 sampling.

    backend_type: 'ideal' (Aer), 'fake' (FakeProvider), or 'real-ibm'.
    backend_method: for 'ideal' -> Aer method name; for 'fake' -> device name.
    '''
    from run_qiskit import run_qiskit_simulation
    res = optimize_qasm(qasm_path, backend_type=backend_type,
                        backend_method=backend_method, verbose=False)
    job = run_qiskit_simulation(res.backend, res.circuit, shots=shots)
    counts = job[0].data.c.get_counts()
    # Qiskit bitstrings are MSB-first — reverse so q0 is first.
    return Counter({k[::-1]: v for k, v in counts.items()})


def run_bloqade_wrapper(qasm_path: str, *, backend_type: str = 'stack',
                        shots: int = SHOTS) -> Counter:
    '''Our Bloqade wrapper (bridge version): return-c kernel, many shots.

    backend_type: 'stack' (StackMemorySimulator) or 'dynamic'
    (DynamicMemorySimulator). Both are PyQrack-backed.
    '''
    from run_bloqade import run_bloqade_simulation
    # res = optimize_qasm_bloqade(qasm_path, backend_type=backend_type,
    #                             parallelize=False, native_gates=False,
    #                             fold=True, verbose=False)
    res = prepare_qasm_for_bloqade(qasm_path, backend_type=backend_type,
                                fold=True, returns = "auto", verbose=False)
    job = run_bloqade_simulation(res.backend, res.kernel, shots=shots)

    counts: Counter = Counter()
    for i in range(len(job)):
        bits = ''.join(str(int(b)) for b in job[i])  # already q0..qN order
        counts[bits] += 1
        
    return counts

def show_histograms(name: str, *runs: tuple) -> None:
    '''Pretty-print aligned histograms from multiple runners.

    Each `run` is (label, Counter). Labels can include backend tags so the
    table header reflects which backend was exercised.
    '''
    all_keys = sorted(set().union(*(run.keys() for _, run in runs)))
    labels = [label for label, _ in runs]
    width = max(20, max(len(l) for l in labels))
    header = f'{"state":<6}  ' + '  '.join(f'{l:>{width}}' for l in labels)
    print(f'\n=== {name} ({SHOTS} shots each) ===')
    print(header)
    print('-' * len(header))
    for k in all_keys:
        row = f'{k:<6}  ' + '  '.join(
            f'{run.get(k,0):>6d} ({100*run.get(k,0)/SHOTS:>4.1f}%)'.rjust(width)
            for _, run in runs
        )
        print(row)

## Test 1 — Bell pair (H + CX)

Classic entanglement smoke test. Expected: roughly 50/50 on `00` and `11`,
nothing else.

In [11]:
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])
print(qc.draw(output='text'))

# Backends: Aer statevector / wrapper-ideal-statevector / Bloqade stack
path = dump_qasm(qc)
try:
    show_histograms('Bell pair',
        ('direct: Aer/statevector',
            run_qiskit_direct(qc, method='statevector')),
        ('wrap: ideal/statevector',
            run_qiskit_wrapper(path, backend_type='ideal',
                               backend_method='statevector')),
        ('bloqade/stack',
            run_bloqade_wrapper(path, backend_type='stack')),
    )
finally:
    os.unlink(path)

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 



=== Bell pair (2048 shots each) ===
state   direct: Aer/statevector  wrap: ideal/statevector            bloqade/stack
---------------------------------------------------------------------------------
00                 1062 (51.9%)             1017 (49.7%)             1010 (49.3%)
11                  986 (48.1%)             1031 (50.3%)             1038 (50.7%)


## Test 2 — 3-qubit GHZ state

Three-qubit generalization of Bell. Expected: roughly 50/50 on `000` and
`111`, nothing else.

In [12]:
qc = QuantumCircuit(3, 3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.measure([0, 1, 2], [0, 1, 2])
print(qc.draw(output='text'))

# Backends: Aer stabilizer / wrapper-ideal-density_matrix / Bloqade dynamic
# GHZ is a pure Clifford circuit, so the stabilizer simulator handles it
# efficiently — much faster than statevector for large Clifford circuits.
path = dump_qasm(qc)
try:
    show_histograms('3-qubit GHZ',
        ('direct: Aer/stabilizer',
            run_qiskit_direct(qc, method='stabilizer')),
        ('wrap: ideal/density_matrix',
            run_qiskit_wrapper(path, backend_type='ideal',
                               backend_method='density_matrix')),
        ('bloqade/dynamic',
            run_bloqade_wrapper(path, backend_type='dynamic')),
    )
finally:
    os.unlink(path)

     ┌───┐          ┌─┐      
q_0: ┤ H ├──■───────┤M├──────
     └───┘┌─┴─┐     └╥┘┌─┐   
q_1: ─────┤ X ├──■───╫─┤M├───
          └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     0  1  2 



=== 3-qubit GHZ (2048 shots each) ===
state       direct: Aer/stabilizer  wrap: ideal/density_matrix             bloqade/dynamic
------------------------------------------------------------------------------------------
000                   1006 (49.1%)                1046 (51.1%)                1069 (52.2%)
111                   1042 (50.9%)                1002 (48.9%)                 979 (47.8%)


## Test 3 — Hadamard superposition over 4 qubits

`H` on every qubit, no entanglement. Expected: a near-uniform distribution
over all 16 outcomes (each outcome ≈ 6.25%).

In [13]:
qc = QuantumCircuit(4, 4)
for q in range(4):
    qc.h(q)
qc.measure(range(4), range(4))
print(qc.draw(output='text'))

# Backends: Aer matrix_product_state / wrapper-ideal-MPS / Bloqade stack
# MPS is well-suited to product states like this — no entanglement.
path = dump_qasm(qc)
try:
    show_histograms('Uniform superposition',
        ('direct: Aer/MPS',
            run_qiskit_direct(qc, method='matrix_product_state')),
        ('wrap: ideal/MPS',
            run_qiskit_wrapper(path, backend_type='ideal',
                               backend_method='matrix_product_state')),
        ('bloqade/stack',
            run_bloqade_wrapper(path, backend_type='stack')),
    )
finally:
    os.unlink(path)

     ┌───┐┌─┐         
q_0: ┤ H ├┤M├─────────
     ├───┤└╥┘┌─┐      
q_1: ┤ H ├─╫─┤M├──────
     ├───┤ ║ └╥┘┌─┐   
q_2: ┤ H ├─╫──╫─┤M├───
     ├───┤ ║  ║ └╥┘┌─┐
q_3: ┤ H ├─╫──╫──╫─┤M├
     └───┘ ║  ║  ║ └╥┘
c: 4/══════╩══╩══╩══╩═
           0  1  2  3 

=== Uniform superposition (2048 shots each) ===
state        direct: Aer/MPS       wrap: ideal/MPS         bloqade/stack
------------------------------------------------------------------------
0000             113 ( 5.5%)           133 ( 6.5%)           122 ( 6.0%)
0001             131 ( 6.4%)           129 ( 6.3%)           137 ( 6.7%)
0010             123 ( 6.0%)           145 ( 7.1%)           123 ( 6.0%)
0011             147 ( 7.2%)           134 ( 6.5%)           140 ( 6.8%)
0100             143 ( 7.0%)           150 ( 7.3%)           134 ( 6.5%)
0101             128 ( 6.2%)           123 ( 6.0%)           127 ( 6.2%)
0110             124 ( 6.1%)           141 ( 6.9%)           118 ( 5.8%)
0111             136 ( 6.6%)           11

## Test 4 — Mixed Clifford + rotation

Exercises single-qubit rotations (`rx`, `rz`) alongside entanglement. The
angle is chosen so the distribution is non-trivial. All three runners should
agree within shot noise.

In [14]:
qc = QuantumCircuit(2, 2)
qc.rx(np.pi / 3, 0)
qc.h(1)
qc.cx(0, 1)
qc.rz(np.pi / 4, 0)
qc.measure([0, 1], [0, 1])
print(qc.draw(output='text'))

# Backends: Aer statevector / wrapper-ideal-automatic / Bloqade dynamic
# Note: stabilizer would fail here because rx/rz are non-Clifford —
# 'automatic' lets Aer pick whichever method is appropriate.
path = dump_qasm(qc)
try:
    show_histograms('Mixed Clifford + rotations',
        ('direct: Aer/statevector',
            run_qiskit_direct(qc, method='statevector')),
        ('wrap: ideal/automatic',
            run_qiskit_wrapper(path, backend_type='ideal',
                               backend_method='automatic')),
        ('bloqade/dynamic',
            run_bloqade_wrapper(path, backend_type='dynamic')),
    )
finally:
    os.unlink(path)

     ┌─────────┐     ┌─────────┐┌─┐
q_0: ┤ Rx(π/3) ├──■──┤ Rz(π/4) ├┤M├
     └──┬───┬──┘┌─┴─┐└───┬─┬───┘└╥┘
q_1: ───┤ H ├───┤ X ├────┤M├─────╫─
        └───┘   └───┘    └╥┘     ║ 
c: 2/═════════════════════╩══════╩═
                          1      0 

=== Mixed Clifford + rotations (2048 shots each) ===
state   direct: Aer/statevector    wrap: ideal/automatic          bloqade/dynamic
---------------------------------------------------------------------------------
00                  741 (36.2%)              789 (38.5%)              777 (37.9%)
01                  782 (38.2%)              273 (13.3%)              750 (36.6%)
10                  266 (13.0%)              734 (35.8%)              261 (12.7%)
11                  259 (12.6%)              252 (12.3%)              260 (12.7%)


## Test 5 — Toffoli (CCX) on the |110⟩ state

Prepare `|110⟩` with `X` gates on q0 and q1, then apply a Toffoli. Expected:
100% on `111` (the Toffoli flips q2 iff q0 and q1 are both 1).

In [15]:
qc = QuantumCircuit(3, 3)
qc.x(0)
qc.x(1)
qc.ccx(0, 1, 2)
qc.measure([0, 1, 2], [0, 1, 2])
print(qc.draw(output='text'))

# Backends: Aer automatic / wrapper-FAKE-fake_manila / Bloqade stack
# Note: stabilizer would fail here — Toffoli (ccx) is NOT a Clifford gate
# (its standard decomposition uses T gates). 'automatic' lets Aer pick.
# The wrapper uses a 5-qubit fake IBM device with realistic noise model
# and coupling map, so its column will spread mass off the ideal '111'.
path = dump_qasm(qc)
try:
    show_histograms('Toffoli on |110>',
        ('direct: Aer/automatic',
            run_qiskit_direct(qc, method='automatic')),
        ('wrap: fake/fake_manila',
            run_qiskit_wrapper(path, backend_type='fake',
                               backend_method='fake_manila')),
        ('bloqade/stack',
            run_bloqade_wrapper(path, backend_type='stack')),
    )
finally:
    os.unlink(path)

     ┌───┐     ┌─┐      
q_0: ┤ X ├──■──┤M├──────
     ├───┤  │  └╥┘┌─┐   
q_1: ┤ X ├──■───╫─┤M├───
     └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ─────┤ X ├─╫──╫─┤M├
          └───┘ ║  ║ └╥┘
c: 3/═══════════╩══╩══╩═
                0  1  2 


/home/dp107543/miniconda3/envs/quantum-testing/lib/python3.13/site-packages/qiskit_ibm_runtime/fake_provider/backends/nighthawk/fake_nighthawk.py:72: UserWarning: Properties of fake_nighthawk are not intended to represent typical nighthawk error values.
  warnings.warn(



=== Toffoli on |110> (2048 shots each) ===
state    direct: Aer/automatic  wrap: fake/fake_manila           bloqade/stack
------------------------------------------------------------------------------
000                  0 ( 0.0%)               2 ( 0.1%)               0 ( 0.0%)
001                  0 ( 0.0%)              23 ( 1.1%)               0 ( 0.0%)
010                  0 ( 0.0%)               9 ( 0.4%)               0 ( 0.0%)
011                  0 ( 0.0%)             288 (14.1%)               0 ( 0.0%)
100                  0 ( 0.0%)               7 ( 0.3%)               0 ( 0.0%)
101                  0 ( 0.0%)              62 ( 3.0%)               0 ( 0.0%)
110                  0 ( 0.0%)              55 ( 2.7%)               0 ( 0.0%)
111              2048 (100.0%)            1602 (78.2%)           2048 (100.0%)


## Interpreting the results

- **Tests 1-4** are noiseless — all three runners should agree up to shot
  noise. Mismatches there are real bugs.
- **Test 5** uses a fake noisy IBM device on the wrap column, so expect that
  column to deviate from the deterministic `111` peak. The other two columns
  should still hit `111` exactly.
- For the **distributional tests** (3 and 4), use column-wise percentages
  to sanity-check shape rather than per-cell counts.

The ordering convention (`q0` leftmost) is normalized across all three
runners by the helper functions. The Aer-method choice per test exercises
the major Aer simulation paths: state-vector, density-matrix, MPS,
stabilizer, and automatic (Aer picks).